# ISIC-2019 — Exploratory Data Analysis

Run after editing `configs/default.yaml` to point at your local dataset.
This notebook does NOT modify any files; it only reads the raw CSVs + images.

Contents:
1. Class distribution & imbalance ratio
2. Source-dataset distribution (HAM / BCN / MSK)
3. Class × source crosstab (where do the rare classes come from?)
4. Sample image grid, per class
5. Metadata missingness (age / sex / anatomical site)
6. Image-size distribution
7. (Optional) Post-preprocessing sanity checks if artifacts exist

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / 'configs' / 'default.yaml').exists():
    if REPO.parent == REPO:
        raise RuntimeError('could not find repo root (configs/default.yaml)')
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print('repo root:', REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from utils.config import load_config, resolve_paths, resolve_image_path
from utils.labels import load_groundtruth, load_metadata, CLASSES

cfg = load_config(REPO / 'configs' / 'default.yaml')
paths = resolve_paths(cfg)
print('data root:', paths.root)
print('work dir :', paths.work_dir)

In [ ]:
gt = load_groundtruth(paths.groundtruth_csv)
meta = load_metadata(paths.metadata_csv)
df = gt.merge(meta, on='image', how='left')
print('rows:', len(df))
df.head()

## 1. Class distribution

Confirms the imbalance ratio. The headline number reviewers want to see is `max_count / min_count`.

In [ ]:
counts = df['class'].value_counts().reindex(CLASSES)
ratio = counts.max() / counts.min()
print(counts.to_string())
print(f'\nimbalance ratio (max/min): {ratio:.1f}x')

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax, color='steelblue')
ax.set_ylabel('image count')
ax.set_xlabel('class')
ax.set_title(f'ISIC-2019 class distribution  (imbalance ratio {ratio:.0f}x)')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

## 2. Source distribution

ISIC-2019 is a union of HAM10000 + BCN_20000 + MSK. Acquisition device differs, which is why we stratify on source in the split.

In [ ]:
src_counts = df['source'].value_counts()
print(src_counts.to_string())

fig, ax = plt.subplots(figsize=(5, 3))
sns.barplot(x=src_counts.index, y=src_counts.values, ax=ax, color='darkorange')
ax.set_ylabel('image count'); ax.set_xlabel('source dataset')
plt.tight_layout(); plt.show()

## 3. Class × source crosstab

Tells us whether a rare class is concentrated in a single source — relevant for generalization.

In [ ]:
ct = pd.crosstab(df['class'], df['source']).reindex(CLASSES)
display(ct)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_title('class × source')
plt.tight_layout(); plt.show()

## 4. Sample image grid — 4 images per class

Quick visual sanity check that label folders are correct and images load.

In [ ]:
rng = np.random.default_rng(int(cfg['seed']))
n_per_class = 4
fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(n_per_class * 2.2, len(CLASSES) * 2.2))
for r, cls in enumerate(CLASSES):
    pool = df.loc[df['class'] == cls, 'image'].to_numpy()
    pick = rng.choice(pool, size=min(n_per_class, len(pool)), replace=False)
    for c in range(n_per_class):
        ax = axes[r, c]
        ax.axis('off')
        if c >= len(pick):
            continue
        p = resolve_image_path(str(pick[c]), cls, paths)
        if p is None:
            ax.set_title(f'{pick[c]}\n(MISSING)', fontsize=7, color='red')
            continue
        with Image.open(p) as im:
            ax.imshow(im.convert('RGB'))
        if c == 0:
            ax.set_ylabel(cls, fontsize=11)
        ax.set_title(pick[c], fontsize=7)
plt.tight_layout(); plt.show()

## 5. Metadata missingness

Important: the proposed model uses metadata via cross-attention. Knowing the missingness rate up front decides how aggressively we need to model 'missing' as its own token vs imputing.

In [ ]:
meta_cols = [c for c in ['age_approx', 'sex', 'anatom_site_general', 'lesion_id'] if c in df.columns]
miss = df[meta_cols].isna().mean().sort_values(ascending=False)
print('Missing fraction per metadata field:')
print((miss * 100).round(1).astype(str) + '%')

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(x=miss.index, y=miss.values * 100, color='crimson', ax=ax)
ax.set_ylabel('% missing'); ax.set_ylim(0, 100)
ax.set_title('metadata missingness')
plt.tight_layout(); plt.show()

In [ ]:
# missingness by class — does any class have systematically less metadata?
miss_by_class = df.groupby('class')[meta_cols].apply(lambda g: g.isna().mean()).reindex(CLASSES)
display((miss_by_class * 100).round(1))

## 6. Image size distribution

Sampled (decoding all 25k just to read shapes is wasteful). Drives the choice of resize strategy.

In [ ]:
sample_n = 500
samp = df.sample(min(sample_n, len(df)), random_state=int(cfg['seed']))
shapes = []
missing = 0
for _, r in samp.iterrows():
    p = resolve_image_path(str(r['image']), str(r['class']), paths)
    if p is None:
        missing += 1
        continue
    with Image.open(p) as im:
        shapes.append(im.size)  # (W, H)
shapes = np.array(shapes)
print(f'sampled {len(shapes)} ({missing} missing).')
print('width  : min/median/max =', shapes[:, 0].min(), int(np.median(shapes[:, 0])), shapes[:, 0].max())
print('height : min/median/max =', shapes[:, 1].min(), int(np.median(shapes[:, 1])), shapes[:, 1].max())
ar = shapes[:, 0] / shapes[:, 1]
print('aspect : min/median/max =', f'{ar.min():.2f} {np.median(ar):.2f} {ar.max():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(shapes[:, 0], bins=40, color='steelblue'); axes[0].set_title('width')
axes[1].hist(ar, bins=40, color='seagreen'); axes[1].set_title('aspect ratio (W/H)')
plt.tight_layout(); plt.show()

## 7. (Optional) Post-preprocessing sanity

If you've already run the preprocessing pipeline, this confirms the memmap is healthy.

In [ ]:
if paths.memmap_images.exists() and paths.memmap_index_csv.exists():
    images = np.load(paths.memmap_images, mmap_mode='r')
    labels = np.load(paths.memmap_labels, mmap_mode='r')
    idx = pd.read_csv(paths.memmap_index_csv)
    print('images:', images.shape, images.dtype)
    print('labels:', labels.shape, labels.dtype)
    print('splits:', idx['split'].value_counts().to_dict())

    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    pick = np.random.default_rng(0).choice(len(images), size=8, replace=False)
    for ax, k in zip(axes.flat, pick):
        ax.imshow(images[k]); ax.axis('off')
        ax.set_title(f"{CLASSES[int(labels[k])]}  row={k}", fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('(no memmap yet — run preprocessing/run_all.py first)')